In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm

In [2]:
data = pd.read_csv('./data/train.csv')
data

,question,context,c_id
0,To whom did the Virgin Mary allegedly appear i...,"Architecturally, the school has a Catholic cha...",0
1,What is in front of the Notre Dame Main Building?,"Architecturally, the school has a Catholic cha...",0
2,The Basilica of the Sacred heart at Notre Dame...,"Architecturally, the school has a Catholic cha...",0
3,What is the Grotto at Notre Dame?,"Architecturally, the school has a Catholic cha...",0
4,What sits on top of the Main Building at Notre...,"Architecturally, the school has a Catholic cha...",0
...,...,...,...
87594,In what US state did Kathmandu first establish...,"Kathmandu Metropolitan City (KMC), in order to...",18890
87595,What was Yangon previously known as?,"Kathmandu Metropolitan City (KMC), in order to...",18890
87596,With what Belorussian city does Kathmandu have...,"Kathmandu Metropolitan City (KMC), in order to...",18890
87597,In what year did Kathmandu create its initial ...,"Kathmandu Metropolitan City (KMC), in order to...",18890


In [3]:
data['c_id'].unique().size

18891

In [4]:
documents = data[['context', 'c_id']].drop_duplicates().reset_index(drop=True)
documents

,context,c_id
0,"Architecturally, the school has a Catholic cha...",0
1,"As at most other universities, Notre Dame's st...",1
2,The university is the major seat of the Congre...,2
3,The College of Engineering was established in ...,3
4,All of Notre Dame's undergraduate students are...,4
...,...,...
18886,"Institute of Medicine, the central college of ...",18886
18887,Football and Cricket are the most popular spor...,18887
18888,The total length of roads in Nepal is recorded...,18888
18889,The main international airport serving Kathman...,18889


In [5]:
from sklearn.feature_extraction.text import CountVectorizer


vectorizer = CountVectorizer(max_features=100)

X = vectorizer.fit_transform(documents['context'])

In [6]:
X.shape, X.dtype, X

((18891, 100),
 dtype('int64'),
 <Compressed Sparse Row sparse matrix of dtype 'int64'
 	with 401893 stored elements and shape (18891, 100)>)

In [7]:
def transform_text(vectorizer, text):
    '''
    Print the text and the vector
    vectorizer: sklearn.vectorizer
    text: str
    '''
    print('Text:', text)
    vector = vectorizer.transform([text])
    vector = vectorizer.inverse_transform(vector)
    print('Vect:', vector)


question = "Where early humans lived?"
transform_text(vectorizer, question)

Text: Where early humans lived?
Vect: [array(['early'], dtype='<U10')]


In [8]:
def retriever(X, Y):
    """
    X: (n_documents, n_features)
    Y: (n_questions, n_features)
    
    Returns: best document index for each question
    """
    similarity = X @ Y.T

    similarity = similarity.toarray()

    best_idx = np.argmax(similarity, axis=0).item()
    
    return best_idx


In [9]:
question = "Where early humans lived?"

y = vectorizer.transform([question])
best_idx = retriever(X, y)

doc = documents.iloc[best_idx]
print(f"Question: {question}")
print(f"Answer:   {doc.context}")

Question: Where early humans lived?
Answer:   Exposure to antibiotics early in life is associated with increased body mass in humans and mouse models. Early life is a critical period for the establishment of the intestinal microbiota and for metabolic development. Mice exposed to subtherapeutic antibiotic treatment (STAT)– with either penicillin, vancomycin, penicillin and vancomycin, or chlortetracycline had altered composition of the gut microbiota as well as its metabolic capabilities. Moreover, research have shown that mice given low-dose penicillin (1 μg/g body weight) around birth and throughout the weaning process had an increased body mass and fat mass, accelerated growth, and increased hepatic expression of genes involved in adipogenesis, compared to controlled mice. In addition, penicillin in combination with a high-fat diet increased fasting insulin levels in mice. However, it is unclear whether or not antibiotics cause obesity in humans. Studies have found a correlation bet

# Submission

In [11]:
test = pd.read_csv('./data/test-docs.csv')
test_questions = pd.read_csv('./data/test-questions.csv')
test

,context
0,China is the country with the largest populati...
1,The bulk of remaining commercial flight offeri...
2,Based on the similar shifts underway the natio...
3,"Since 2004, through the V&A + RIBA Architectur..."
4,"Society in the Japanese ""Tokugawa period"" (Edo..."
...,...
18886,"On July 1, 1985, Gorbachev promoted Eduard She..."
18887,Richmond is home to the rapidly developing Vir...
18888,"Portugal has the largest aquarium in Europe, t..."
18889,"At times a character may ""turn"", altering thei..."


In [12]:
test_questions

,question
0,What is the Grotto at Notre Dame?
1,What sits on top of the Main Building at Notre...
2,What is the daily student paper at Notre Dame ...
3,In what year did the student paper Common Sens...
4,Which prize did Frederick Buechner create?
...,...
37777,"As of 2004, how many kilometers of road existe..."
37778,What is Nepal's primary airport for internatio...
37779,"Starting in the center of Kathmandu, how many ..."
37780,With what Belorussian city does Kathmandu have...


In [13]:
X_test = vectorizer.transform(test['context'])

Y_test = vectorizer.transform(test_questions['question'])

In [14]:
X_test.shape, Y_test.shape

((18891, 100), (37782, 100))

In [15]:
similarity = X_test @ Y_test.T

In [16]:
similarity = similarity.toarray()

In [17]:
similarity.shape

(18891, 37782)

In [18]:
pred_ids = []
for q_idx, question in tqdm(enumerate(test_questions.question), total=len(test_questions)):
    best_idx = np.argmax(similarity[:, q_idx]).item()

    pred_ids.append(best_idx)


100%|██████████| 37782/37782 [00:01<00:00, 27762.67it/s]


In [19]:
submission = pd.DataFrame({'id': test_questions.index.tolist(), "pred_id": pred_ids})
submission.to_csv("sample_submission.csv", index=False)
